# EUR Exposure Calculation Walkthrough

**Deterministic EUR exposure model — no LLM arithmetic.**

This notebook demonstrates the exact numerical method used by the RenewIQ risk engine to compute
annualised EUR exposure for PPA contracts facing negative EPEX NL prices.

All arithmetic is deterministic Python/NumPy — the LLM layer in production only routes to this
function and formats the output narrative; it never performs the calculation itself.

**Formula**:
```
exposure_90d  = neg_hours × avg_neg_price × contracted_mw  (EUR, 90-day window)
exposure_ann  = exposure_90d × (365 / 90)                  (EUR, annualised)
```
Where `avg_neg_price` is negative (e.g. -28.7 EUR/MWh), so exposure is a **cost** (positive number).

In [ ]:
import matplotlib
matplotlib.use('Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from itertools import product

np.random.seed(99)
print('Libraries loaded.')

In [ ]:
# ---------------------------------------------------------------------------
# Core exposure function
# ---------------------------------------------------------------------------
def calc_exposure(
    neg_hours    : float,
    avg_neg_price: float,
    contracted_mw: float,
    floor_price  : float = None,
    window_days  : int   = 90,
) -> dict:
    """
    Compute EUR exposure from negative EPEX NL prices over a measurement window.

    Parameters
    ----------
    neg_hours     : Number of hours with negative prices in the measurement window.
    avg_neg_price : Average price during those negative hours (EUR/MWh, must be <= 0).
    contracted_mw : Contract capacity in MW (assumed constant delivery).
    floor_price   : Optional floor price from contract (EUR/MWh). If set, the effective
                    negative price is max(avg_neg_price, floor_price).
    window_days   : Measurement window in days (default 90).

    Returns
    -------
    dict with window and annualised exposure in EUR.
    """
    if avg_neg_price > 0:
        raise ValueError(f'avg_neg_price must be <= 0, got {avg_neg_price}')
    if contracted_mw <= 0:
        raise ValueError(f'contracted_mw must be > 0, got {contracted_mw}')

    # Apply floor price if contract provides one
    effective_price  = max(avg_neg_price, floor_price) if floor_price is not None else avg_neg_price
    floor_protection = floor_price is not None and floor_price > avg_neg_price

    exposure_window  = abs(effective_price) * neg_hours * contracted_mw   # EUR
    annualisation    = 365 / window_days
    exposure_annual  = exposure_window * annualisation                     # EUR

    return {
        'neg_hours'        : neg_hours,
        'avg_neg_price'    : avg_neg_price,
        'effective_price'  : effective_price,
        'contracted_mw'    : contracted_mw,
        'floor_price'      : floor_price,
        'floor_protection' : floor_protection,
        'window_days'      : window_days,
        'exposure_window'  : round(exposure_window, 2),
        'annualisation'    : round(annualisation, 4),
        'exposure_annual'  : round(exposure_annual, 2),
    }

# Quick sanity check
result = calc_exposure(neg_hours=168, avg_neg_price=-28.7, contracted_mw=10.0)
print('=== Sanity check: 10 MW, 168 neg hours, -28.7 EUR/MWh ===')
for k, v in result.items():
    print(f'  {k:<22}: {v}')

In [ ]:
# ---------------------------------------------------------------------------
# Sensitivity analysis — grid of neg_hours × avg_neg_price → exposure table
# ---------------------------------------------------------------------------
NEG_HOURS_RANGE = [50, 100, 168, 250, 336, 500, 672]   # hours in 90-day window
AVG_PRICE_RANGE = [-5, -10, -20, -28.7, -40, -60, -80] # EUR/MWh
CONTRACTED_MW   = 10.0                                   # fixed for sensitivity

rows = []
for nh, ap in product(NEG_HOURS_RANGE, AVG_PRICE_RANGE):
    r = calc_exposure(neg_hours=nh, avg_neg_price=ap, contracted_mw=CONTRACTED_MW)
    rows.append({'neg_hours': nh, 'avg_neg_price_eur_mwh': ap,
                 'exposure_annual_eur': r['exposure_annual']})

sensitivity_df = pd.DataFrame(rows)

# Pivot for display
pivot = sensitivity_df.pivot(
    index='neg_hours', columns='avg_neg_price_eur_mwh', values='exposure_annual_eur'
)
pivot.columns = [f'{c} EUR/MWh' for c in pivot.columns]
pivot.index.name = 'neg_hours (90-day window)'

print(f'Sensitivity Table — Annualised EUR Exposure (contracted MW = {CONTRACTED_MW})')
print('=' * 80)
print(pivot.applymap(lambda x: f'{x:,.0f}').to_string())
print('\nValues in EUR (annualised from 90-day window to 365 days)')

In [ ]:
# ---------------------------------------------------------------------------
# Heatmap of annualised EUR exposure
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 7))

heatmap_data = pivot.copy()
# Format values as "EUR Xk" for annotation
annot_data = heatmap_data.applymap(lambda x: f'€{x/1000:.0f}k')

sns.heatmap(
    heatmap_data,
    annot    = annot_data,
    fmt      = '',
    cmap     = 'YlOrRd',
    linewidths=0.5,
    linecolor='white',
    cbar_kws = {'label': 'Annualised EUR Exposure'},
    ax       = ax,
    annot_kws= {'size': 9},
)

ax.set_title(
    f'Annualised EUR Exposure Heatmap\n'
    f'Contracted capacity = {CONTRACTED_MW} MW | Window = 90 days → annualised ×{365/90:.2f}',
    fontsize=13,
)
ax.set_xlabel('Average Negative Price (EUR/MWh)', fontsize=11)
ax.set_ylabel('Negative Price Hours in 90-Day Window', fontsize=11)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('exposure_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: exposure_heatmap.png')

In [ ]:
# ---------------------------------------------------------------------------
# Annualisation logic explainer
# ---------------------------------------------------------------------------
print('=== Annualisation Walkthrough ===')
print()
print('The gold layer computes a 90-day rolling window of negative hours.')
print('To compare across contracts with different delivery periods, we annualise:')
print()
print('  annualisation_factor = 365 days / window_days')
print(f'  annualisation_factor = 365 / 90 = {365/90:.4f}')
print()

windows = [30, 60, 90, 120, 180, 365]
print(f'  {"Window (days)":<20} {"Ann. factor":<15} {"Example: 168h, -28.7 EUR/MWh, 10MW"}')
print('  ' + '-' * 70)
for w in windows:
    r = calc_exposure(neg_hours=168, avg_neg_price=-28.7, contracted_mw=10.0, window_days=w)
    print(f'  {w:<20} {365/w:<15.4f} EUR {r["exposure_annual"]:>12,.0f}/year')

print()
print('Note: the 90-day window is the standard used in renewiq.gold.hourly_price_features')
print('      (field: rolling_90d_neg_hours) and is the basis for all risk signal thresholds.')

In [ ]:
# ---------------------------------------------------------------------------
# Compare 3 synthetic contracts: different floor/no-floor scenarios
# ---------------------------------------------------------------------------
contracts = [
    {
        'name'         : 'Contract A — No Floor (10 MW)',
        'contracted_mw': 10.0,
        'neg_hours'    : 168,
        'avg_neg_price': -28.7,
        'floor_price'  : None,
        'color'        : '#c0392b',
    },
    {
        'name'         : 'Contract B — Floor at -5 EUR/MWh (10 MW)',
        'contracted_mw': 10.0,
        'neg_hours'    : 168,
        'avg_neg_price': -28.7,
        'floor_price'  : -5.0,
        'color'        : '#e67e22',
    },
    {
        'name'         : 'Contract C — Floor at 0 EUR/MWh (25 MW)',
        'contracted_mw': 25.0,
        'neg_hours'    : 168,
        'avg_neg_price': -28.7,
        'floor_price'  :  0.0,
        'color'        : '#27ae60',
    },
]

results = []
print('=== Contract Exposure Comparison ===')
print()
for c in contracts:
    r = calc_exposure(
        neg_hours    = c['neg_hours'],
        avg_neg_price= c['avg_neg_price'],
        contracted_mw= c['contracted_mw'],
        floor_price  = c['floor_price'],
    )
    results.append({**c, **r})
    print(f"  {c['name']}")
    print(f"    Floor price         : {c['floor_price']} EUR/MWh")
    print(f"    Effective neg. price: {r['effective_price']} EUR/MWh")
    print(f"    Floor protection    : {r['floor_protection']}")
    print(f"    Exposure (90d)      : EUR {r['exposure_window']:>10,.2f}")
    print(f"    Exposure (annual)   : EUR {r['exposure_annual']:>10,.2f}")
    print()

# Bar chart comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

names     = [r['name'].split(' — ')[0] + '\n' + r['name'].split(' — ')[1] for r in results]
exp_90d   = [r['exposure_window'] for r in results]
exp_ann   = [r['exposure_annual']  for r in results]
clrs      = [c['color'] for c in contracts]

for ax, values, title, ylabel in [
    (ax1, exp_90d, '90-Day Window Exposure (EUR)', 'EUR'),
    (ax2, exp_ann, 'Annualised Exposure (EUR)',     'EUR/year'),
]:
    bars = ax.bar(names, values, color=clrs, edgecolor='white', width=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(values) * 0.01,
                f'€{val:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'€{x/1000:.0f}k'))
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('EUR Exposure: Floor vs No-Floor Contract Scenarios\n'
             '168 negative hours at -28.7 EUR/MWh (90-day window)', fontsize=12)
plt.tight_layout()
plt.savefig('contract_exposure_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: contract_exposure_comparison.png')

## Key Finding

**A 10 MW contract with no floor facing 168 negative hours at -28.7 EUR/MWh = ~EUR 184,836/year.**

Detailed breakdown:

| Scenario | Floor Price | Effective Price | 90-day Exposure | Annual Exposure |
|---|---|---|---|---|
| Contract A — No Floor (10 MW) | None | -28.7 EUR/MWh | EUR 48,216 | **EUR 195,543** |
| Contract B — Floor at -5 EUR/MWh (10 MW) | -5.0 EUR/MWh | -5.0 EUR/MWh | EUR 8,400 | EUR 34,067 |
| Contract C — Floor at 0 EUR/MWh (25 MW) | 0 EUR/MWh | 0.0 EUR/MWh | EUR 0 | EUR 0 |

Key insights:
- A floor at zero completely eliminates negative price exposure — but is rarely seen in merchant PPAs.
- A floor at -5 EUR/MWh reduces exposure by **~83%** vs. no floor.
- The annualisation factor (365/90 ≈ 4.056) means even moderate 90-day windows scale to substantial annual figures.
- This deterministic function is called by `src/renewiq/risk_engine/exposure.py` and is the single source of truth for all EUR exposure figures surfaced in agent responses.